# Visualizing Latent Space Evolution During Training

How does an AI actually "learn" to beat physical radio noise? 

In this notebook, we run an intensive training loop directly in the browser. Every few epochs, we will freeze the network, extract its multi-dimensional `Embedding` matrix, and use **Principal Component Analysis (PCA)** to squash it down to 2D. 

We will watch the words physically push themselves apart in space to avoid overlapping when the Gaussian noise hits them!

In [ ]:
%matplotlib inline
import sys
import os
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
import numpy as np

# Add src to path
sys.path.append(os.path.abspath(r'C:\Users\Shrish\Desktop\semantic-comm\actual_project'))
from src.tokenizer import SemanticTokenizer
from src.model import JointSemanticModel

print("Libraries loaded successfully.")

### Step 1: Setup the Toy Dataset and Model
To make the visualization clear, we will use a tiny vocabulary of just a few sentences and train the model intensively on them.

In [ ]:
sentences = [
    "the quick brown fox jumps over the lazy dog",
    "semantic communication is the future of telecommunications",
    "artificial intelligence bypasses shannon limits natively",
    "deploy the autonomous sensor network immediately"
] * 10  # Duplicate to create a small batch

tokenizer = SemanticTokenizer(min_freq=1)
tokenizer.fit(sentences)

# Convert to tensors
encoded_data = [tokenizer.encode(s, max_length=10) for s in sentences]
dataset = torch.tensor(encoded_data, dtype=torch.long)

model = JointSemanticModel(vocab_size=tokenizer.vocab_size, embed_dim=32, hidden_dim=64, snr_db=5.0)
optimizer = optim.Adam(model.parameters(), lr=0.005)
criterion = nn.CrossEntropyLoss(ignore_index=tokenizer.PAD_ID)

print(f"Dataset prepared. Vocab size: {tokenizer.vocab_size}")

### Step 2: The Training Loop with PCA Snapshots
We will train the model for **200 Epochs**. At Epoch 0, 50, 100, and 200, we will extract the model's `Embedding` weights and save them.

In [ ]:
epochs = 200
snapshots = {}
snapshot_epochs = [0, 50, 100, 199]

print("Beginning Training Loop...")
for epoch in range(epochs):
    # Save snapshot
    if epoch in snapshot_epochs:
        embeddings = model.encoder.embedding.weight.detach().numpy()
        snapshots[epoch] = embeddings.copy()

    model.train()
    optimizer.zero_grad()
    
    outputs = model(dataset, dataset)
    outputs_flat = outputs[:, :-1, :].reshape(-1, tokenizer.vocab_size)
    targets_flat = dataset[:, 1:].reshape(-1)
    
    loss = criterion(outputs_flat, targets_flat)
    loss.backward()
    optimizer.step()
    
    if epoch % 50 == 0:
        print(f"Epoch {epoch} | Loss: {loss.item():.4f}")
        
# Save final snapshot
embeddings = model.encoder.embedding.weight.detach().numpy()
snapshots[199] = embeddings.copy()
print(f"Epoch 200 | Loss: {loss.item():.4f}")
print("Training Complete!")

### Step 3: Visualizing the Semantic Big Bang
We apply PCA to reduce the 32-dimensional embedding space down to 2 dimensions so we can plot it on an X/Y graph. 

Notice how at **Epoch 0**, all the words are clustered together in a random blob. The noise from the `AWGNChannel` easily overlaps them, causing hallucinations.

As training progresses, the neural network learns to physically **push the word vectors away from each other** in space. This guarantees that even if a word is hit with heavy Gaussian noise, it won't cross the boundary into another word's territory!

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
fig.suptitle("Evolution of Word Embeddings Combating Channel Noise", fontsize=16)

for i, ep in enumerate(snapshot_epochs):
    pca = PCA(n_components=2)
    reduced_embeddings = pca.fit_transform(snapshots[ep])
    
    # We ignore the first 4 tokens (PAD, UNK, BOS, EOS) to just see actual words
    axes[i].scatter(reduced_embeddings[4:, 0], reduced_embeddings[4:, 1], color='crimson', alpha=0.7, edgecolors='k')
    axes[i].set_title(f"Epoch {ep}")
    axes[i].grid(True, linestyle='--', alpha=0.5)
    
    # Lock the axes so the expansion is visually obvious
    axes[i].set_xlim(-4, 4)
    axes[i].set_ylim(-4, 4)

plt.show()